In [1]:
from pathlib import Path
import random
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import FEMTO_DIR, FEMTO_FS, REPORTS_DIR, SEED, WINDOW_SIZE
from src.data.loader import load_femto_bearing
from src.visualization.plots import plot_fft_comparison, plot_raw_signal

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
FIGURES_DIR = REPORTS_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
bearing_path = FEMTO_DIR / "10. FEMTO Bearing" / "Training_set" / "Learning_set" / "Bearing1_1"
bearing_frame = load_femto_bearing(bearing_path)
bearing_name = bearing_path.name

In [3]:
file_count = bearing_frame["file_idx"].nunique()
rows_per_file = bearing_frame.groupby("file_idx").size()
total_duration_minutes = file_count * 10 / 60

print(f"files: {file_count}")
print(f"rows per file: {rows_per_file.unique().tolist()}")
print(f"columns: {bearing_frame.columns.tolist()}")
print(f"duration minutes: {total_duration_minutes:.2f}")

files: 2803
rows per file: [2560]
columns: ['hour', 'min', 'sec', 'microsec', 'acc_h', 'acc_v', 'file_idx']
duration minutes: 467.17


In [4]:
plot_raw_signal(
    bearing_frame,
    "acc_h",
    FIGURES_DIR / "femto_bearing1_1_acc_h_mean_amplitude.png",
)

In [5]:
rms_by_file = bearing_frame.groupby("file_idx")[["acc_h", "acc_v"]].agg(
    lambda values: np.sqrt(np.mean(values.to_numpy() ** 2))
)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(rms_by_file.index, rms_by_file["acc_h"], label="Horizontal RMS", linewidth=1.2)
ax.plot(rms_by_file.index, rms_by_file["acc_v"], label="Vertical RMS", linewidth=1.2)
ax.set_xlabel("File index (× 10s)")
ax.set_ylabel("RMS acceleration")
ax.set_title("Bearing1_1 Per-File RMS")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "femto_bearing1_1_rms_by_channel.png", dpi=150)
plt.close(fig)

In [6]:
healthy_window = bearing_frame.loc[bearing_frame["file_idx"] == 0, "acc_h"].to_numpy()[:WINDOW_SIZE]
degraded_window = bearing_frame.loc[bearing_frame["file_idx"] == file_count - 1, "acc_h"].to_numpy()[:WINDOW_SIZE]
time_axis_ms = np.arange(WINDOW_SIZE) / FEMTO_FS * 1_000

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(time_axis_ms, healthy_window, label="Healthy file 0", linewidth=1.0)
ax.plot(time_axis_ms, degraded_window, label=f"Late file {file_count - 1}", linewidth=1.0)
ax.set_xlabel("Time (ms)")
ax.set_ylabel("Horizontal acceleration")
ax.set_title("Early vs Late FEMTO Bearing1_1 Window")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "femto_bearing1_1_early_late_window.png", dpi=150)
plt.close(fig)

In [7]:
plot_fft_comparison(
    healthy_window,
    degraded_window,
    FEMTO_FS,
    FIGURES_DIR / "femto_bearing1_1_early_late_fft.png",
)

The horizontal vibration amplitude grows strongly near the end of life: in this local copy, first-file horizontal RMS is about 0.56 while the last-file value is about 5.61. The late-life waveform also has much larger peaks, so simple energy features should be competitive. The FFT comparison is still useful because degradation may shift energy into different bands, which motivates combining time, frequency, and envelope features before testing whether the LSTM autoencoder adds value beyond RMS-based baselines.